## Silver Layer — Titanic

**Purpose:** Cleansed, conformed, analyst-friendly. One row per passenger.

**Transformations:**
- Parse Title out of Name (Mr / Mrs / Miss / Master / Other)
- Derive AgeGroup and FamilySize
- Map single-letter codes (Sex, Embarked) to readable values
- Map Pclass to "First" / "Second" / "Third"
- Extract CabinDeck from first letter of Cabin
- Impute Age using median by (Pclass, Sex) — a standard Titanic trick that's better than a global median
- Basic DQ checks (fail fast on surprises)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_table = "bronze_titanic"
silver_table = "silver_passenger"

df = spark.table(bronze_table)

In [ ]:
# --- Derive Title from Name --------------------------------------------------
# Name format: "Surname, Title. Given Names"
df = df.withColumn("Title", F.trim(F.regexp_extract(F.col("Name"), r",\s*([^\.]+)\.", 1)))

# Collapse rare titles into sensible buckets
title_map = {
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Lady": "Rare", "Countess": "Rare", "the Countess": "Rare",
    "Capt": "Rare", "Col": "Rare", "Don": "Rare", "Dr": "Rare",
    "Major": "Rare", "Rev": "Rare", "Sir": "Rare", "Jonkheer": "Rare",
}
mapping_expr = F.create_map([F.lit(x) for kv in title_map.items() for x in kv])
df = df.withColumn(
    "Title",
    F.when(F.col("Title").isin(list(title_map.keys())), mapping_expr[F.col("Title")])
     .otherwise(F.col("Title"))
)
# Anything unexpected -> "Other"
df = df.withColumn(
    "Title",
    F.when(F.col("Title").isin("Mr", "Mrs", "Miss", "Master", "Rare"), F.col("Title"))
     .otherwise("Other")
)

In [ ]:
# --- Impute Age: median per (Pclass, Sex) -----------------------------------
# Using percentile_approx in a window gives each row its group's median.
w = Window.partitionBy("Pclass", "Sex")
df = df.withColumn(
    "Age",
    F.when(F.col("Age").isNull(),
           F.percentile_approx(F.col("Age"), 0.5).over(w))
     .otherwise(F.col("Age"))
)
# Safety net: any group still null -> overall median
overall_median = df.approxQuantile("Age", [0.5], 0.01)[0]
df = df.withColumn("Age", F.coalesce(F.col("Age"), F.lit(overall_median)))

In [ ]:
# --- Readable mappings -------------------------------------------------------
df = df.withColumn(
    "SexDescription",
    F.when(F.col("Sex") == "male",   "Male")
     .when(F.col("Sex") == "female", "Female")
     .otherwise("Unknown")
)

df = df.withColumn(
    "EmbarkedPort",
    F.when(F.col("Embarked") == "S", "Southampton")
     .when(F.col("Embarked") == "C", "Cherbourg")
     .when(F.col("Embarked") == "Q", "Queenstown")
     .otherwise("Unknown")
)

df = df.withColumn(
    "PassengerClass",
    F.when(F.col("Pclass") == 1, "First")
     .when(F.col("Pclass") == 2, "Second")
     .when(F.col("Pclass") == 3, "Third")
     .otherwise("Unknown")
)

In [ ]:
# --- Derivations -------------------------------------------------------------
df = df.withColumn("FamilySize",  F.col("SibSp") + F.col("Parch") + F.lit(1))
df = df.withColumn("IsAlone",     (F.col("FamilySize") == 1).cast("int"))
df = df.withColumn(
    "CabinDeck",
    F.when((F.col("Cabin").isNotNull()) & (F.col("Cabin") != ""),
           F.substring(F.col("Cabin"), 1, 1))
     .otherwise("Unknown")
)

df = df.withColumn(
    "AgeGroup",
    F.when(F.col("Age") < 13,  "Child")
     .when(F.col("Age") < 20,  "Teen")
     .when(F.col("Age") < 40,  "Adult")
     .when(F.col("Age") < 60,  "MiddleAged")
     .otherwise("Senior")
)

df = df.withColumn("Survived", F.col("Survived").cast("boolean"))

In [ ]:
# --- Final projection --------------------------------------------------------
df_silver = df.select(
    F.col("PassengerId"),
    F.col("Name"),
    F.col("Title"),
    F.col("SexDescription").alias("Sex"),
    F.col("Age"),
    F.col("AgeGroup"),
    F.col("PassengerClass"),
    F.col("Pclass").alias("PassengerClassNumber"),
    F.col("SibSp").alias("SiblingsSpousesAboard"),
    F.col("Parch").alias("ParentsChildrenAboard"),
    F.col("FamilySize"),
    F.col("IsAlone"),
    F.col("Ticket"),
    F.col("Fare"),
    F.col("Cabin"),
    F.col("CabinDeck"),
    F.col("EmbarkedPort"),
    F.col("Survived"),
    F.col("_ingested_at"),
    F.col("_pipeline_run_id"),
).withColumn("_processed_at", F.current_timestamp())

In [ ]:
# --- Data-quality gates ------------------------------------------------------
total = df_silver.count()
dupes = df_silver.groupBy("PassengerId").count().filter("count > 1").count()
null_ids = df_silver.filter(F.col("PassengerId").isNull()).count()

print(f"Silver rows: {total} | duplicate PassengerIds: {dupes} | null IDs: {null_ids}")

assert dupes == 0,    "DQ failure: duplicate PassengerIds in Silver"
assert null_ids == 0, "DQ failure: null PassengerIds in Silver"
assert total > 0,     "DQ failure: Silver is empty"

In [ ]:
(
    df_silver.write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(silver_table)
)

print(f"Silver table '{silver_table}' written.")
spark.sql(f"SELECT * FROM {silver_table} LIMIT 5").show(truncate=False)